# 📰 Fake News Detection with DeBERTa-v3 (Optimized for Low-VRAM)

Bu proje, metin tabanlı haber içeriklerinin gerçekliğini analiz etmek için geliştirilmiş bir **İkili Sınıflandırma (Binary Classification)** modelidir.

---

## 📊 Veri Seti (`GonzaloA/fake_news`)

* **İçerik ve Boyut:** Veri seti, farklı kategorilerden derlenmiş on binlerce haber metni içerir. Model, analizlerini haberin gövde metni (`text` sütunu) üzerinden yapar.
* **Etiketleme Sistemi:** Veriler `0` (Gerçek Haber) ve `1` (Sahte Haber) olarak etiketlenmiştir.
* **Neden Bu Veri Seti Seçildi?** Veri seti sınıflar açısından tamamen **dengeli (balanced)** bir yapıya sahiptir.

---

## 🧠 Model Mimarisi ve "Dikkat" (Attention) Mekanizması

Projede geleneksel Transformer modelleri yerine, **DeBERTa-v3** (Decoding-enhanced BERT with disentangled attention) modeli tercih edilmiştir.

### ❌ Eski Yöntem: Standart Self-Attention (Karışık Zihinli Analist)
Eski modellerde, yapay zeka bir kelimeyi incelerken kelimenin **Sözlük Anlamı** ile **Cümledeki Sırasını (Konumunu)** birbirine yapıştırarak tek bir paket haline getirir.

### ✅ Bu Projedeki Yöntem: Disentangled Attention (Uzmanlar Masası)
DeBERTa modeli ise "Ayrıştırılmış Dikkat" kullanır. Karar verirken iki farklı "sanal uzman" çalıştırır:
1. **Anlam Uzmanı:** Sadece kelimelerin saf sözlük anlamına odaklanır.
2. **Mesafe Uzmanı:** Kelimelerin anlamını bilmez, sadece aralarındaki boşluğu sayar.

---

## 🛠️ Mimari Kararlar ve 6GB VRAM Optimizasyonu

1. **Mixed Precision Training (fp16)**
2. **Gradient Accumulation** (Sanal Batch Size = 16)
3. **1 Nöronlu Çıktı ve Sigmoid + BCEWithLogitsLoss**

---

## 🚀 Kurulum

```bash
pip install torch transformers datasets scikit-learn matplotlib sentencepiece protobuf tqdm
```

# 🔍 Fake News Detection with DeBERTa-v3-base

**Architecture:** `microsoft/deberta-v3-base` + Single-Neuron Classification Head  
**Loss Function:** `BCEWithLogitsLoss` (binary, logit-space)  
**Hardware Target:** 6 GB VRAM (fp16 mixed precision, batch_size=4, grad_accum=4)  
**Dataset:** `GonzaloA/fake_news` from Hugging Face Hub  

---

In [1]:
# ============================================================
# CELL 1 — Imports & Setup
# ============================================================
# Uncomment to install dependencies on first run:
# !pip install torch transformers datasets scikit-learn matplotlib sentencepiece protobuf tqdm

import os
import random
import warnings
import numpy as np
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader

# tqdm.auto picks the best display (notebook widget in Jupyter, plain bar in terminal)
from tqdm.auto import tqdm

from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from datasets import load_dataset, ClassLabel

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix, roc_curve
)

from IPython.display import clear_output, display

warnings.filterwarnings('ignore')

# ── Reproducibility ──────────────────────────────────────────
SEED = 42

def set_seed(seed: int = SEED):
    """Fix all random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

# ── Device ───────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'   GPU    : {torch.cuda.get_device_name(0)}')
    print(f'   VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# ── Hyper-parameters (centralised) ───────────────────────────
MODEL_NAME          = 'microsoft/deberta-v3-base'
MAX_LENGTH          = 256
BATCH_SIZE          = 4          # per_device_train_batch_size
GRAD_ACCUM_STEPS    = 4          # effective batch = 4 × 4 = 16
LEARNING_RATE       = 3e-5
WEIGHT_DECAY        = 0.01
NUM_EPOCHS          = 3
WARMUP_RATIO        = 0.10       # first 10 % of steps = warmup
OUTPUT_DIR          = './outputs'

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'✅ Output directory : {OUTPUT_DIR}')
print('✅ Cell 1 complete — imports & seed fixed.')

✅ Device : cuda
   GPU    : NVIDIA GeForce RTX 3060 Laptop GPU
   VRAM   : 6.4 GB
✅ Output directory : ./outputs
✅ Cell 1 complete — imports & seed fixed.


In [2]:
# ============================================================
# CELL 2 — Dataset Preparation
# ============================================================

# ── 2-A  Download the dataset ────────────────────────────────
print('⏳ Loading GonzaloA/fake_news from Hugging Face Hub …')
raw = load_dataset('GonzaloA/fake_news')
print(raw)

# ── FIX: cast 'label' from plain Value(int64) to ClassLabel for stratified split
train_full = raw['train']
train_full = train_full.cast_column(
    'label', ClassLabel(names=['real', 'fake'])   # 0 = real, 1 = fake
)

# Drop rows where 'text' is None / empty (avoids tokenizer errors)
train_full = train_full.filter(
    lambda ex: ex['text'] is not None and len(str(ex['text']).strip()) > 0
)

# Stratified 85 / 15 split
split    = train_full.train_test_split(test_size=0.15, seed=SEED, stratify_by_column='label')
train_hf = split['train']
val_hf   = split['test']

print(f'Train rows : {len(train_hf):,}   |   Val rows : {len(val_hf):,}')
print(f'Label distribution (train) : {dict(zip(*np.unique(train_hf["label"], return_counts=True)))}')

# ── 2-B  Tokenizer ───────────────────────────────────────────
print(f'\n⏳ Loading tokenizer : {MODEL_NAME}')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(text: str) -> dict:
    """Tokenise a single string; returns tensors ready for the model."""
    return tokenizer(
        text,
        max_length=MAX_LENGTH,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )

# ── 2-C  PyTorch Dataset ─────────────────────────────────────
class FakeNewsDataset(Dataset):
    """
    Wraps a Hugging Face dataset split and returns tokenised
    (input_ids, attention_mask, label) tuples.
    """

    def __init__(self, hf_split):
        self.texts  = hf_split['text']
        self.labels = hf_split['label']

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        enc = tokenize(str(self.texts[idx]))
        return {
            'input_ids'      : enc['input_ids'].squeeze(0),
            'attention_mask' : enc['attention_mask'].squeeze(0),
            'label'          : torch.tensor(self.labels[idx], dtype=torch.float32)
        }

# ── 2-D  DataLoaders ─────────────────────────────────────────
# IMPORTANT — Windows + Jupyter: num_workers MUST be 0.
# Spawned child processes cannot import the notebook's global state and crash.
train_dataset = FakeNewsDataset(train_hf)
val_dataset   = FakeNewsDataset(val_hf)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE,
    shuffle=True, num_workers=0,
    pin_memory=(DEVICE.type == 'cuda')
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE * 2,
    shuffle=False, num_workers=0,
    pin_memory=(DEVICE.type == 'cuda')
)

print(f'\n✅ Cell 2 complete.')
print(f'   Train batches : {len(train_loader):,}   |   Val batches : {len(val_loader):,}')

⏳ Loading GonzaloA/fake_news from Hugging Face Hub …


Repo card metadata block was not found. Setting CardData to empty.


DatasetDict({
    train: Dataset({
        features: ['Unnamed: 0', 'title', 'text', 'label'],
        num_rows: 24353
    })
    validation: Dataset({
        features: ['Unnamed: 0', 'title', 'text', 'label'],
        num_rows: 8117
    })
    test: Dataset({
        features: ['Unnamed: 0', 'title', 'text', 'label'],
        num_rows: 8117
    })
})
Train rows : 20,699   |   Val rows : 3,653
Label distribution (train) : {np.int64(0): np.int64(9483), np.int64(1): np.int64(11216)}

⏳ Loading tokenizer : microsoft/deberta-v3-base

✅ Cell 2 complete.
   Train batches : 5,175   |   Val batches : 457


In [3]:
# ============================================================
# CELL 3 — Model Initialization
# ============================================================
# Architecture:
#   DeBERTa-v3-base (768-dim) → Dropout → Linear(768 → 1)
#   Raw logit → BCEWithLogitsLoss during training
#   Sigmoid applied ONLY at evaluation
#
# IMPORTANT — FP32 enforcement:
#   Mixed-precision training (autocast) requires model parameters to be
#   float32.  The GradScaler.unscale_() call will raise
#   "Attempting to unscale FP16 gradients" if any parameter is float16.
#   Some HuggingFace checkpoints ship weights in float16 — we force float32
#   both at load time (torch_dtype) and after construction (.float()).

class FakeNewsClassifier(nn.Module):
    """DeBERTa-v3-base + single-neuron binary classification head."""

    def __init__(self, model_name: str, dropout_p: float = 0.1):
        super().__init__()
        # Force float32 at load time so mixed-precision training works correctly
        self.backbone   = AutoModel.from_pretrained(
            model_name,
            torch_dtype=torch.float32,   # ← ensures weights are FP32, not FP16
        )
        hidden_size     = self.backbone.config.hidden_size  # 768
        self.dropout    = nn.Dropout(p=dropout_p)
        self.classifier = nn.Linear(hidden_size, 1)  # ← 1 neuron only
        nn.init.normal_(self.classifier.weight, std=0.02)
        nn.init.zeros_(self.classifier.bias)

    def forward(self, input_ids, attention_mask):
        outputs  = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        cls_repr = outputs.last_hidden_state[:, 0, :]    # [CLS] token
        cls_repr = self.dropout(cls_repr)
        logits   = self.classifier(cls_repr).squeeze(-1)  # (batch,)
        return logits


print(f'⏳ Loading {MODEL_NAME} …')
model = FakeNewsClassifier(MODEL_NAME)
model = model.float()   # ← secondary safety net: cast ALL params to float32
model = model.to(DEVICE)

# Verify no parameter is in a non-float32 dtype
fp16_params = [(n, p.dtype) for n, p in model.named_parameters() if p.dtype != torch.float32]
if fp16_params:
    print(f'  ⚠️  Found non-float32 params (will cause GradScaler error): {fp16_params[:3]}')
else:
    print('  ✅ All parameters confirmed float32 — GradScaler safe')

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'   Total params     : {total_params:,}')
print(f'   Trainable params : {trainable_params:,}')
print(f'   Classification head : Linear(768 → 1)')
print(f'✅ Cell 3 complete — model on {DEVICE}.')

`torch_dtype` is deprecated! Use `dtype` instead!


⏳ Loading microsoft/deberta-v3-base …


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  ✅ All parameters confirmed float32 — GradScaler safe
   Total params     : 183,832,321
   Trainable params : 183,832,321
   Classification head : Linear(768 → 1)
✅ Cell 3 complete — model on cuda.


In [4]:
# ============================================================
# CELL 4 — Custom Training Loop with Live Visualisation + tqdm
# ============================================================
# Progress tracking:
#  • Outer tqdm bar  → tracks epochs  (total = NUM_EPOCHS)
#  • Inner tqdm bar  → tracks batches (total = len(train_loader) per epoch)
#  • Val tqdm bar    → tracks val batches during evaluation
#  • Live matplotlib plot updated after every epoch

# ── Loss, Optimizer, Scheduler ───────────────────────────────
loss_fn = nn.BCEWithLogitsLoss()

no_decay = ['bias', 'LayerNorm.weight', 'layernorm.weight']
optimizer_groups = [
    {
        'params': [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)],
        'weight_decay': WEIGHT_DECAY
    },
    {
        'params': [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)],
        'weight_decay': 0.0
    }
]
optimizer = AdamW(optimizer_groups, lr=LEARNING_RATE)

total_optimizer_steps = (len(train_loader) // GRAD_ACCUM_STEPS) * NUM_EPOCHS
warmup_steps          = int(WARMUP_RATIO * total_optimizer_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_optimizer_steps
)

# Modern torch.amp API (avoids DeprecationWarning on PyTorch >= 2.0)
scaler = torch.amp.GradScaler(device='cuda')

print(f'Total optimizer steps : {total_optimizer_steps}')
print(f'Warmup steps          : {warmup_steps}')

# ── History containers ────────────────────────────────────────
history = {
    'train_loss' : [],
    'val_loss'   : [],
    'val_acc'    : [],
    'val_f1'     : []
}

# ── Live plot helper ──────────────────────────────────────────
def live_plot(history: dict, epoch: int, num_epochs: int, metrics: dict):
    """Clear Jupyter output and re-render loss curves + metrics table."""
    clear_output(wait=True)

    fig = plt.figure(figsize=(14, 5))
    gs  = gridspec.GridSpec(1, 2, figure=fig)
    epochs_done = list(range(1, len(history['train_loss']) + 1))

    # Loss curves
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.plot(epochs_done, history['train_loss'], 'o-',  color='#4F86F7', label='Train Loss', lw=2)
    ax1.plot(epochs_done, history['val_loss'],   's--', color='#F7714F', label='Val Loss',   lw=2)
    ax1.set_title('Training & Validation Loss', fontsize=13, fontweight='bold')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('BCEWithLogitsLoss')
    ax1.legend()
    ax1.set_xticks(epochs_done)
    ax1.grid(alpha=0.3)

    # Metrics table
    ax2 = fig.add_subplot(gs[0, 1])
    ax2.axis('off')
    rows = []
    for e, (tl, vl, va, vf) in enumerate(
        zip(history['train_loss'], history['val_loss'],
            history['val_acc'],    history['val_f1']), start=1
    ):
        marker = ' ◀' if e == epoch else ''
        rows.append([f'Epoch {e}{marker}', f'{tl:.4f}', f'{vl:.4f}', f'{va:.4f}', f'{vf:.4f}'])

    tbl = ax2.table(
        cellText=rows,
        colLabels=['Epoch', 'Train Loss', 'Val Loss', 'Val Acc', 'Val F1'],
        loc='center', cellLoc='center'
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(10)
    tbl.scale(1.2, 1.6)
    ax2.set_title('Epoch Metrics', fontsize=13, fontweight='bold')

    plt.suptitle(
        f'Fake News Detection — DeBERTa-v3-base   [Epoch {epoch}/{num_epochs}]',
        fontsize=14, fontweight='bold', y=1.02
    )
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'live_training_plot.png'), bbox_inches='tight', dpi=100)
    plt.show()
    plt.close(fig)

    print('─' * 60)
    print(f'  Epoch {epoch}/{num_epochs}')
    print(f'  Train Loss : {metrics["train_loss"]:.6f}')
    print(f'  Val   Loss : {metrics["val_loss"]:.6f}')
    print(f'  Val   Acc  : {metrics["val_acc"]:.4f}')
    print(f'  Val   F1   : {metrics["val_f1"]:.4f}')
    print('─' * 60)


# ── Training function ─────────────────────────────────────────
def train_one_epoch(model, loader, optimizer, scheduler, scaler, loss_fn,
                    grad_accum_steps, device, epoch_idx, num_epochs):
    """Run one full training epoch with a per-batch tqdm progress bar."""
    model.train()
    total_loss  = 0.0
    num_batches = len(loader)
    optimizer.zero_grad()

    # ── Inner tqdm bar (per-batch) ────────────────────────────
    batch_bar = tqdm(
        loader,
        desc=f'  Epoch {epoch_idx}/{num_epochs} [Train]',
        unit='batch',
        leave=True,
        dynamic_ncols=True
    )

    for step, batch in enumerate(batch_bar):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['label'].to(device)

        # autocast: temporarily computes CUDA ops in FP16 for speed/memory
        # Parameters remain in FP32 — gradients also accumulate in FP32
        with torch.amp.autocast(device_type='cuda', enabled=(device.type == 'cuda')):
            logits = model(input_ids, attention_mask)
            loss   = loss_fn(logits, labels)
            loss   = loss / grad_accum_steps

        scaler.scale(loss).backward()
        total_loss += loss.item() * grad_accum_steps

        if (step + 1) % grad_accum_steps == 0 or (step + 1) == num_batches:
            # unscale_: converts scaled FP32 grads back to true FP32 magnitudes
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

        # Update postfix with running avg loss and current LR
        current_lr  = scheduler.get_last_lr()[0]
        running_avg = total_loss / (step + 1)
        batch_bar.set_postfix(
            loss=f'{running_avg:.4f}',
            lr=f'{current_lr:.2e}'
        )

    batch_bar.close()
    return total_loss / num_batches


@torch.no_grad()
def evaluate(model, loader, loss_fn, device, epoch_idx, num_epochs):
    """
    Evaluate on a DataLoader with a per-batch tqdm progress bar.
    Sigmoid is applied HERE (inference only) — never during training.
    """
    model.eval()
    total_loss = 0.0
    all_logits = []
    all_labels = []

    # ── Val tqdm bar ──────────────────────────────────────────
    val_bar = tqdm(
        loader,
        desc=f'  Epoch {epoch_idx}/{num_epochs} [Val  ]',
        unit='batch',
        leave=True,
        dynamic_ncols=True
    )

    for batch in val_bar:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['label'].to(device)

        with torch.amp.autocast(device_type='cuda', enabled=(device.type == 'cuda')):
            logits = model(input_ids, attention_mask)
            loss   = loss_fn(logits, labels)

        total_loss += loss.item()
        all_logits.append(logits.cpu())
        all_labels.append(labels.cpu())

    val_bar.close()

    all_logits = torch.cat(all_logits)
    all_labels = torch.cat(all_labels).numpy()

    # Sigmoid applied ONLY here — never inside forward()
    all_probs = torch.sigmoid(all_logits).numpy()
    all_preds = (all_probs >= 0.5).astype(int)

    avg_loss = total_loss / len(loader)
    acc      = accuracy_score(all_labels, all_preds)
    f1       = f1_score(all_labels, all_preds, average='binary')

    return avg_loss, acc, f1, all_preds, all_probs, all_labels.astype(int)


# ── Main Training Loop ────────────────────────────────────────
best_val_f1     = 0.0
best_model_path = os.path.join(OUTPUT_DIR, 'best_model.pt')

print('🚀 Starting training …\n')

# ── Outer tqdm bar (per-epoch) ────────────────────────────────
epoch_bar = tqdm(
    range(1, NUM_EPOCHS + 1),
    desc='Overall Epochs',
    unit='epoch',
    leave=True,
    dynamic_ncols=True
)

for epoch in epoch_bar:

    # — Train
    train_loss = train_one_epoch(
        model, train_loader, optimizer, scheduler,
        scaler, loss_fn, GRAD_ACCUM_STEPS, DEVICE,
        epoch_idx=epoch, num_epochs=NUM_EPOCHS
    )

    # — Validate
    val_loss, val_acc, val_f1, val_preds, val_probs, val_labels = evaluate(
        model, val_loader, loss_fn, DEVICE,
        epoch_idx=epoch, num_epochs=NUM_EPOCHS
    )

    # Update outer epoch bar with summary metrics
    epoch_bar.set_postfix(
        train_loss=f'{train_loss:.4f}',
        val_loss=f'{val_loss:.4f}',
        val_f1=f'{val_f1:.4f}'
    )

    # — Record history
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_f1'].append(val_f1)

    epoch_metrics = dict(
        train_loss=train_loss, val_loss=val_loss,
        val_acc=val_acc,        val_f1=val_f1
    )

    # — Live loss plot
    live_plot(history, epoch, NUM_EPOCHS, epoch_metrics)

    # — Save best checkpoint
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), best_model_path)
        print(f'  💾 New best model saved (Val F1 = {best_val_f1:.4f})')

epoch_bar.close()

print('\n✅ Cell 4 complete — training finished.')
print(f'   Best Val F1 : {best_val_f1:.4f}')
print(f'   Best model  : {best_model_path}')

────────────────────────────────────────────────────────────
  Epoch 3/3
  Train Loss : 0.033976
  Val   Loss : 0.040390
  Val   Acc  : 0.9836
  Val   F1   : 0.9846
────────────────────────────────────────────────────────────
  💾 New best model saved (Val F1 = 0.9846)

✅ Cell 4 complete — training finished.
   Best Val F1 : 0.9846
   Best model  : ./outputs\best_model.pt


In [5]:
# ============================================================
# CELL 5 — Final Evaluation on the Validation Set
# ============================================================

print('⏳ Loading best model weights …')
model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
model.to(DEVICE)
model.eval()

# Full inference pass (val tqdm bar shown)
_, final_acc, final_f1, final_preds, final_probs, final_labels = evaluate(
    model, val_loader, loss_fn, DEVICE,
    epoch_idx='Final', num_epochs='—'
)

precision = precision_score(final_labels, final_preds, average='binary')
recall    = recall_score(final_labels, final_preds, average='binary')
f1        = f1_score(final_labels, final_preds, average='binary')
accuracy  = accuracy_score(final_labels, final_preds)
roc_auc   = roc_auc_score(final_labels, final_probs)

print('\n' + '═' * 50)
print('       FINAL EVALUATION — VALIDATION SET')
print('═' * 50)
print(f'  Accuracy  : {accuracy:.4f}  ({accuracy*100:.2f} %)')
print(f'  Precision : {precision:.4f}')
print(f'  Recall    : {recall:.4f}')
print(f'  F1 Score  : {f1:.4f}')
print(f'  ROC-AUC   : {roc_auc:.4f}')
print('═' * 50)

eval_results = dict(
    accuracy=accuracy, precision=precision,
    recall=recall,     f1=f1,
    roc_auc=roc_auc
)

print('\n✅ Cell 5 complete — final metrics computed.')

⏳ Loading best model weights …


  Epoch Final/— [Val  ]:   0%|          | 0/457 [00:00<?, ?batch/s]


══════════════════════════════════════════════════
       FINAL EVALUATION — VALIDATION SET
══════════════════════════════════════════════════
  Accuracy  : 0.9836  (98.36 %)
  Precision : 0.9995
  Recall    : 0.9702
  F1 Score  : 0.9846
  ROC-AUC   : 0.9989
══════════════════════════════════════════════════

✅ Cell 5 complete — final metrics computed.


In [6]:
# ============================================================
# CELL 6 — Visualisations & Saving
# ============================================================

plt.style.use('seaborn-v0_8-darkgrid')
epochs_x = list(range(1, NUM_EPOCHS + 1))

# ── 6-A  Final Loss Curves ────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(epochs_x, history['train_loss'], 'o-',  color='#4F86F7', lw=2.5, label='Train Loss', markersize=8)
ax.plot(epochs_x, history['val_loss'],   's--', color='#F7714F', lw=2.5, label='Val Loss',   markersize=8)
ax.fill_between(epochs_x, history['train_loss'], history['val_loss'], alpha=0.1, color='#9B59B6')
ax.set_title('Training & Validation Loss — DeBERTa-v3-base', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('BCEWithLogitsLoss', fontsize=12)
ax.set_xticks(epochs_x)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.4)
loss_path = os.path.join(OUTPUT_DIR, 'final_loss_curves.png')
plt.tight_layout()
plt.savefig(loss_path, dpi=150, bbox_inches='tight')
plt.show()
plt.close()
print(f'✅ Loss curves saved → {loss_path}')

# ── 6-B  Confusion Matrix ─────────────────────────────────────
cm = confusion_matrix(final_labels, final_preds)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
plt.colorbar(im, ax=ax)
class_names = ['Real (0)', 'Fake (1)']
tick_marks  = np.arange(len(class_names))
ax.set_xticks(tick_marks);  ax.set_xticklabels(class_names, fontsize=11)
ax.set_yticks(tick_marks);  ax.set_yticklabels(class_names, fontsize=11)
ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_ylabel('True Label',      fontsize=12)
ax.set_title('Confusion Matrix', fontsize=14, fontweight='bold')
thresh = cm.max() / 2.0
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, f'{cm[i, j]:,}', ha='center', va='center', fontsize=13,
                color='white' if cm[i, j] > thresh else 'black')
cm_path = os.path.join(OUTPUT_DIR, 'confusion_matrix.png')
plt.tight_layout()
plt.savefig(cm_path, dpi=150, bbox_inches='tight')
plt.show()
plt.close()
print(f'✅ Confusion matrix saved → {cm_path}')

# ── 6-C  ROC Curve ───────────────────────────────────────────
fpr, tpr, _ = roc_curve(final_labels, final_probs)
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, color='#4F86F7', lw=2.5, label=f'ROC curve (AUC = {eval_results["roc_auc"]:.4f})')
ax.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random classifier')
ax.fill_between(fpr, tpr, alpha=0.12, color='#4F86F7')
ax.set_title('Receiver Operating Characteristic', fontsize=14, fontweight='bold')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate',  fontsize=12)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.4)
roc_path = os.path.join(OUTPUT_DIR, 'roc_curve.png')
plt.tight_layout()
plt.savefig(roc_path, dpi=150, bbox_inches='tight')
plt.show()
plt.close()
print(f'✅ ROC curve saved → {roc_path}')

# ── 6-D  Save Final Model Weights ────────────────────────────
final_weights_path = os.path.join(OUTPUT_DIR, 'deberta_fake_news_final.pt')
torch.save(model.state_dict(), final_weights_path)
print(f'✅ Final model weights saved → {final_weights_path}')

# ── 6-E  Summary ─────────────────────────────────────────────
print('\n' + '═' * 55)
print(' 🏁  TRAINING COMPLETE — SUMMARY')
print('═' * 55)
print(f'  Model          : {MODEL_NAME}')
print(f'  Max Token Len  : {MAX_LENGTH}')
print(f'  Effective Batch: {BATCH_SIZE * GRAD_ACCUM_STEPS} ({BATCH_SIZE} × {GRAD_ACCUM_STEPS})')
print(f'  Epochs trained : {NUM_EPOCHS}')
print(f'  Best Val F1    : {best_val_f1:.4f}')
print(f'  Final Accuracy : {eval_results["accuracy"]:.4f}')
print(f'  Final ROC-AUC  : {eval_results["roc_auc"]:.4f}')
print('═' * 55)
print(f'\n  Outputs saved to: {os.path.abspath(OUTPUT_DIR)}')
print('\n✅ Cell 6 complete — all artefacts saved.')

✅ Loss curves saved → ./outputs\final_loss_curves.png
✅ Confusion matrix saved → ./outputs\confusion_matrix.png
✅ ROC curve saved → ./outputs\roc_curve.png
✅ Final model weights saved → ./outputs\deberta_fake_news_final.pt

═══════════════════════════════════════════════════════
 🏁  TRAINING COMPLETE — SUMMARY
═══════════════════════════════════════════════════════
  Model          : microsoft/deberta-v3-base
  Max Token Len  : 256
  Effective Batch: 16 (4 × 4)
  Epochs trained : 3
  Best Val F1    : 0.9846
  Final Accuracy : 0.9836
  Final ROC-AUC  : 0.9989
═══════════════════════════════════════════════════════

  Outputs saved to: c:\Users\efeka\Documents\AI\AI and data science\LastTango\outputs

✅ Cell 6 complete — all artefacts saved.


In [8]:
# ============================================================
# CELL 7 — Custom Inference on Sample Texts (Real vs Fake)
# ============================================================
import os
import torch

print("🔍 Loading final model weights for custom inference...")
inference_model = FakeNewsClassifier(MODEL_NAME)
inference_model.load_state_dict(torch.load(os.path.join(OUTPUT_DIR, 'deberta_fake_news_final.pt'), map_location=DEVICE))
inference_model.to(DEVICE)
inference_model.eval()

sample_texts = [
    # Metin 1: Standart Gazetecilik Yapısı
    "WASHINGTON (Reuters) - The U.S. Senate voted on Thursday to approve the new defense budget, aiming to restructure current military expenditures following weeks of intense bipartisan negotiations.",
    
    # Metin 2: Manipülatif Clickbait Yapısı
    "BREAKING: You will not believe what they just found! Shocking new leaked documents completely destroy the official narrative. The mainstream media is trying to cover this up immediately!"
]

print("\n" + "═" * 65)
print(" 🕵️‍♂️  CUSTOM TEXT INFERENCE RESULTS (FROM SAVED WEIGHTS)")
print("═" * 65)

for i, text in enumerate(sample_texts):
    encoded = tokenizer(
        text, max_length=MAX_LENGTH, padding='max_length',
        truncation=True, return_tensors='pt'
    )
    
    input_ids = encoded['input_ids'].to(DEVICE)
    attention_mask = encoded['attention_mask'].to(DEVICE)
    
    with torch.no_grad():
        with torch.amp.autocast(device_type='cuda', enabled=(DEVICE.type == 'cuda')):
            logit = inference_model(input_ids, attention_mask)
            prob = torch.sigmoid(logit).item() 
            
    # DÜZELTİLEN KISIM: Veri setinde 1 = Real, 0 = Fake
    prediction = 1 if prob >= 0.5 else 0
    label_str = "REAL (1)" if prediction == 1 else "FAKE (0)"
    
    confidence = prob if prediction == 1 else (1 - prob)
    
    print(f"📄 Text Snippet {i+1}: '{text}'")
    print(f"🧠 Raw Logit   : {logit.item():.4f}")
    print(f"📊 Probability : {prob:.4f} (Probability of being REAL)")
    print(f"🎯 Prediction  : {label_str} (Confidence: {confidence*100:.2f}%)")
    print("─" * 65)

print("✅ Cell 7 complete — manual verification successful.")

🔍 Loading final model weights for custom inference...


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



═════════════════════════════════════════════════════════════════
 🕵️‍♂️  CUSTOM TEXT INFERENCE RESULTS (FROM SAVED WEIGHTS)
═════════════════════════════════════════════════════════════════
📄 Text Snippet 1: 'WASHINGTON (Reuters) - The U.S. Senate voted on Thursday to approve the new defense budget, aiming to restructure current military expenditures following weeks of intense bipartisan negotiations.'
🧠 Raw Logit   : 11.0781
📊 Probability : 1.0000 (Probability of being REAL)
🎯 Prediction  : REAL (1) (Confidence: 100.00%)
─────────────────────────────────────────────────────────────────
📄 Text Snippet 2: 'BREAKING: You will not believe what they just found! Shocking new leaked documents completely destroy the official narrative. The mainstream media is trying to cover this up immediately!'
🧠 Raw Logit   : -9.5938
📊 Probability : 0.0001 (Probability of being REAL)
🎯 Prediction  : FAKE (0) (Confidence: 99.99%)
─────────────────────────────────────────────────────────────────
✅ Cell 7 c